# Notebook 12: Feature Remediation

Removes ambiguous gap features and recalculates inactivity using a zero-ambiguity approach. Runs LOYOCV for four baseline models to confirm feature quality after the fix.

In [28]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler

RESULTS_DIR = os.path.join(os.getcwd(), "Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

## STEP 1: Load the Modeling Dataset

In [29]:
# Load datasets created in Notebooks 1 & 2
modeling_data = pd.read_excel("modeling_dataset.xlsx")
final_dataset = pd.read_excel("final_dataset.xlsx")

print(f"Modeling dataset shape: {modeling_data.shape}")
print(f"\nOutcome distribution:")
print(modeling_data["outcome"].value_counts())
print(f"\nClass proportions:")
print((modeling_data["outcome"].value_counts() / len(modeling_data) * 100).round(1))

# Separate features and outcome
y = modeling_data["outcome"]
X_old = modeling_data.drop(columns=["year", "outcome"]).copy()

print(f"\nOriginal feature set: {X_old.shape[1]} features")
print(f"Features: {X_old.columns.tolist()}")

Modeling dataset shape: (1154, 30)

Outcome distribution:
outcome
1    887
0    267
Name: count, dtype: int64

Class proportions:
outcome
1    76.9
0    23.1
Name: count, dtype: float64

Original feature set: 28 features
Features: ['num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w', 'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w', 'gap5to6_3w', 'gap6to7_3w', 'gap7to8_3w', 'total_inactivity_3w']


## STEP 2: Remove Ambiguous Gap Features

In [30]:
# Identify problematic features
gap_cols_to_remove = [
    'gap1to2_3w', 'gap2to3_3w', 'gap3to4_3w', 'gap4to5_3w',
    'gap5to6_3w', 'gap6to7_3w', 'gap7to8_3w',
    'total_inactivity_3w'  # This is also ambiguous
]

print("Removing these ambiguous features:")
for col in gap_cols_to_remove:
    status = "✓" if col in X_old.columns else "✗ (not found)"
    print(f"  {status} {col}")

X_no_gaps = X_old.drop(columns=gap_cols_to_remove, errors='ignore').copy()

print(f"\nFeature count after removal: {X_no_gaps.shape[1]}")
print(f"Columns remaining: {X_no_gaps.columns.tolist()}")

Removing these ambiguous features:
  ✓ gap1to2_3w
  ✓ gap2to3_3w
  ✓ gap3to4_3w
  ✓ gap4to5_3w
  ✓ gap5to6_3w
  ✓ gap6to7_3w
  ✓ gap7to8_3w
  ✓ total_inactivity_3w

Feature count after removal: 20
Columns remaining: ['num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w']


## STEP 3: Recalculate Inactivity with Zero-Ambiguity Resolution

In [31]:
# Load original attempt-level data to recalculate last_completion_week
# ── UPDATE THIS PATH to point to your mastery_data.xlsx file ──────────────
file_path = "mastery_data.xlsx"  # relative path - place file in same folder as this notebook
# ────────────────────────────────────────────────────────────────────────────

attempts_df = pd.read_excel(file_path, sheet_name="All Attempts")

# Rename columns
attempts_df = attempts_df.rename(columns={
    'new ID': 'id',
    'year': 'year',
    'progression': 'level',
    'pass/fail': 'pass_fail',
    'attempt_number': 'attempt_num',
    'week_number': 'week'
})

attempts_df['id'] = attempts_df['id'].str.strip()

# Filter to weeks 1-3 only
early_attempts = attempts_df[attempts_df['week'] <= 3].copy()

print(f"Early attempts (weeks 1-3): {len(early_attempts)} records")
print(f"Unique student-cohort instances: {early_attempts[['id', 'year']].drop_duplicates().shape[0]}")

Early attempts (weeks 1-3): 3642 records
Unique student-cohort instances: 1133


## STEP 4: Calculate `last_completion_week` and `trailing_gap_weeks`

In [32]:
# Get master student list
master_students = final_dataset[['id', 'year']].drop_duplicates().copy()

# Calculate last completion week (latest week in which student completed ANY level)
last_completion = (
    early_attempts[early_attempts['pass_fail'] == 1]
    .groupby(['id', 'year'])['week']
    .max()
    .reset_index()
    .rename(columns={'week': 'last_completion_week'})
)

# Merge with all students, filling missing with 0 (no completions)
last_completion_full = master_students.merge(
    last_completion,
    on=['id', 'year'],
    how='left'
).fillna(0)

last_completion_full['last_completion_week'] = last_completion_full['last_completion_week'].astype(int)

# Calculate trailing_gap_weeks: weeks of silence from last completion to end of week 3
last_completion_full['trailing_gap_weeks'] = (
    3 - last_completion_full['last_completion_week']
)

print("Feature calculation examples:")
print("\nStudent A (rapid progression): last_completion_week = 3")
print("  → trailing_gap_weeks = 3 - 3 = 0")
print("  → Result: Low inactivity ✓")

print("\nStudent B (one level week 1, silence): last_completion_week = 1")
print("  → trailing_gap_weeks = 3 - 1 = 2")
print("  → Result: Medium-high inactivity ✓")

print("\nStudent C (completely inactive): last_completion_week = 0")
print("  → trailing_gap_weeks = 3 - 0 = 3")
print("  → Result: Highest inactivity ✓")

print(f"\nDistribution of trailing_gap_weeks:")
print(last_completion_full['trailing_gap_weeks'].value_counts().sort_index())

Feature calculation examples:

Student A (rapid progression): last_completion_week = 3
  → trailing_gap_weeks = 3 - 3 = 0
  → Result: Low inactivity ✓

Student B (one level week 1, silence): last_completion_week = 1
  → trailing_gap_weeks = 3 - 1 = 2
  → Result: Medium-high inactivity ✓

Student C (completely inactive): last_completion_week = 0
  → trailing_gap_weeks = 3 - 0 = 3
  → Result: Highest inactivity ✓

Distribution of trailing_gap_weeks:
trailing_gap_weeks
0    722
1    252
2    144
3     36
Name: count, dtype: int64


## STEP 5: Construct `total_inactivity_adjusted`

In [33]:
# Get the old total_inactivity_3w (sum of internal gaps between levels)
# Add student identifiers so we can merge safely on ['id', 'year']
old_inactivity = X_old[['total_inactivity_3w']].copy()
# 'id' is not in modeling_data - pulling from final_dataset which retains it
old_inactivity['id']   = final_dataset['id'].values
old_inactivity['year'] = final_dataset['year'].values

# Merge explicitly on ['id', 'year'] - avoids silent positional
# misalignment if the two DataFrames happen to be in different row orders
inactivity_merged = old_inactivity.merge(
    last_completion_full[['id', 'year', 'trailing_gap_weeks']],
    on=['id', 'year'],
    how='left'
)

# Align the merged result back to the original row order
# Use final_dataset for ['id', 'year'] since modeling_data drops the id column
inactivity_merged = final_dataset[['id', 'year']].merge(
    inactivity_merged, on=['id', 'year'], how='left'
)

total_inactivity_adjusted = (
    inactivity_merged['total_inactivity_3w'] +
    inactivity_merged['trailing_gap_weeks']
).values

print(f"total_inactivity_adjusted statistics:")
print(f"  Min: {total_inactivity_adjusted.min():.1f}")
print(f"  Max: {total_inactivity_adjusted.max():.1f}")
print(f"  Mean: {total_inactivity_adjusted.mean():.2f}")
print(f"  Std: {total_inactivity_adjusted.std():.2f}")

print(f"\nComparison: old vs new inactivity")
comparison = pd.DataFrame({
    'Old (ambiguous)': old_inactivity['total_inactivity_3w'].values[:10],
    'Last completion week': last_completion_full['last_completion_week'].values[:10],
    'Trailing gap': last_completion_full['trailing_gap_weeks'].values[:10],
    'New (adjusted)': total_inactivity_adjusted[:10]
})
print(comparison.to_string())

total_inactivity_adjusted statistics:
  Min: 0.0
  Max: 3.0
  Mean: 1.80
  Std: 0.55

Comparison: old vs new inactivity
   Old (ambiguous)  Last completion week  Trailing gap  New (adjusted)
0                2                     3             0               2
1                2                     3             0               2
2                2                     3             0               2
3                2                     3             0               2
4                2                     3             0               2
5                2                     3             0               2
6                2                     3             0               2
7                0                     1             2               2
8                2                     3             0               2
9                2                     3             0               2


## STEP 6: Create New Feature Matrix

In [34]:
# Create new feature matrix
X_new = X_no_gaps.copy()

# Add the new inactivity feature
X_new['total_inactivity_adjusted'] = total_inactivity_adjusted

# Add reference columns (not used in modeling but useful for analysis)
X_new['last_completion_week'] = last_completion_full['last_completion_week'].values
X_new['trailing_gap_weeks'] = last_completion_full['trailing_gap_weeks'].values

print(f"New feature matrix shape: {X_new.shape}")
print(f"\nFeature summary:")
print(f"  Removed 8 ambiguous features (7 gap + total_inactivity_3w)")
print(f"  Added 3 new features (total_inactivity_adjusted + references)")
print(f"  Net change: {X_new.shape[1]} - {X_old.shape[1]} = {X_new.shape[1] - X_old.shape[1]} columns")

print(f"\nFeatures for modeling ({len(X_new.columns)} total):")
for i, col in enumerate(X_new.columns, 1):
    print(f"  {i:2d}. {col}")

New feature matrix shape: (1154, 23)

Feature summary:
  Removed 8 ambiguous features (7 gap + total_inactivity_3w)
  Added 3 new features (total_inactivity_adjusted + references)
  Net change: 23 - 28 = -5 columns

Features for modeling (23 total):
   1. num_atmp_l1
   2. num_atmp_l2
   3. num_atmp_l3
   4. num_atmp_l4
   5. num_atmp_l5
   6. num_atmp_l6
   7. num_atmp_l7
   8. num_atmp_l8
   9. num_atmp_l11
  10. wk_comp_l1
  11. wk_comp_l2
  12. wk_comp_l3
  13. wk_comp_l4
  14. wk_comp_l5
  15. wk_comp_l6
  16. wk_comp_l7
  17. wk_comp_l8
  18. total_attempts_3w
  19. levels_completed_3w
  20. efficiency_3w
  21. total_inactivity_adjusted
  22. last_completion_week
  23. trailing_gap_weeks


## STEP 7: Prepare for Cross-Validation

In [35]:
# Define features for modeling (exclude reference columns)
feature_cols_modeling = [
    col for col in X_new.columns 
    if col not in ['last_completion_week', 'trailing_gap_weeks']
]

X_model = X_new[feature_cols_modeling].copy()
year_col = modeling_data['year'].copy()

print(f"Modeling feature set: {len(feature_cols_modeling)} features")
print(f"Modeling features: {feature_cols_modeling}")
print(f"\nReady for Leave-One-Year-Out Cross-Validation")
print(f"Test years: {sorted(year_col.unique())}")

Modeling feature set: 21 features
Modeling features: ['num_atmp_l1', 'num_atmp_l2', 'num_atmp_l3', 'num_atmp_l4', 'num_atmp_l5', 'num_atmp_l6', 'num_atmp_l7', 'num_atmp_l8', 'num_atmp_l11', 'wk_comp_l1', 'wk_comp_l2', 'wk_comp_l3', 'wk_comp_l4', 'wk_comp_l5', 'wk_comp_l6', 'wk_comp_l7', 'wk_comp_l8', 'total_attempts_3w', 'levels_completed_3w', 'efficiency_3w', 'total_inactivity_adjusted']

Ready for Leave-One-Year-Out Cross-Validation
Test years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018)]


## STEP 8: Leave-One-Year-Out Cross-Validation

In [36]:
# Store results for all models
results_all = {
    'Logistic Regression': [],
    'Random Forest': [],
    'Gradient Boosting': [],
    'SVM': []
}

test_years = sorted(year_col.dropna().unique())

for test_year in test_years:
    print(f"\n{'='*70}")
    print(f"TEST YEAR: {int(test_year)}")
    print(f"{'='*70}")
    
    # Split by year
    test_mask = year_col == test_year
    X_train = X_model.loc[~test_mask].copy()
    X_test = X_model.loc[test_mask].copy()
    y_train = y.loc[~test_mask].copy()
    y_test = y.loc[test_mask].copy()
    
    print(f"Train size: {len(X_train):4d} | Test size: {len(X_test):3d}")
    print(f"Train fail rate: {(y_train==0).sum()/len(y_train)*100:.1f}%")
    print(f"Test fail rate:  {(y_test==0).sum()/len(y_test)*100:.1f}%")
    
    # ==============================
    # LOGISTIC REGRESSION
    # ==============================
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr_model.fit(X_train_scaled, y_train)
    
    y_pred_lr = lr_model.predict(X_test_scaled)
    y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
    
    results_all['Logistic Regression'].append({
        'test_year': int(test_year),
        'accuracy': accuracy_score(y_test, y_pred_lr),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_lr),
        'auc': roc_auc_score(y_test, y_prob_lr),
        'recall_pass': recall_score(y_test, y_pred_lr, pos_label=1),
        'f1_pass': f1_score(y_test, y_pred_lr, pos_label=1),
        'recall_fail': recall_score(y_test, y_pred_lr, pos_label=0),
        'f1_fail': f1_score(y_test, y_pred_lr, pos_label=0),
    })
    
    lr_res = results_all['Logistic Regression'][-1]
    print(f"\nLogistic Regression:")
    print(f"  Acc: {lr_res['accuracy']:.3f} | Bal.Acc: {lr_res['balanced_accuracy']:.3f} | AUC: {lr_res['auc']:.3f}")
    print(f"  Recall(Fail): {lr_res['recall_fail']:.3f} | F1(Fail): {lr_res['f1_fail']:.3f}")
    
    # ==============================
    # RANDOM FOREST
    # ==============================
    rf_model = RandomForestClassifier(
        n_estimators=200, max_depth=None, min_samples_leaf=1,
        class_weight='balanced', random_state=42, n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    y_pred_rf = rf_model.predict(X_test)
    y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
    
    results_all['Random Forest'].append({
        'test_year': int(test_year),
        'accuracy': accuracy_score(y_test, y_pred_rf),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_rf),
        'auc': roc_auc_score(y_test, y_prob_rf),
        'recall_pass': recall_score(y_test, y_pred_rf, pos_label=1),
        'f1_pass': f1_score(y_test, y_pred_rf, pos_label=1),
        'recall_fail': recall_score(y_test, y_pred_rf, pos_label=0),
        'f1_fail': f1_score(y_test, y_pred_rf, pos_label=0),
    })
    
    rf_res = results_all['Random Forest'][-1]
    print(f"\nRandom Forest:")
    print(f"  Acc: {rf_res['accuracy']:.3f} | Bal.Acc: {rf_res['balanced_accuracy']:.3f} | AUC: {rf_res['auc']:.3f}")
    print(f"  Recall(Fail): {rf_res['recall_fail']:.3f} | F1(Fail): {rf_res['f1_fail']:.3f}")
    
    # ==============================
    # GRADIENT BOOSTING
    # ==============================
    gb_model = GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
    )
    gb_model.fit(X_train, y_train)
    
    y_pred_gb = gb_model.predict(X_test)
    y_prob_gb = gb_model.predict_proba(X_test)[:, 1]
    
    results_all['Gradient Boosting'].append({
        'test_year': int(test_year),
        'accuracy': accuracy_score(y_test, y_pred_gb),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_gb),
        'auc': roc_auc_score(y_test, y_prob_gb),
        'recall_pass': recall_score(y_test, y_pred_gb, pos_label=1),
        'f1_pass': f1_score(y_test, y_pred_gb, pos_label=1),
        'recall_fail': recall_score(y_test, y_pred_gb, pos_label=0),
        'f1_fail': f1_score(y_test, y_pred_gb, pos_label=0),
    })
    
    gb_res = results_all['Gradient Boosting'][-1]
    print(f"\nGradient Boosting:")
    print(f"  Acc: {gb_res['accuracy']:.3f} | Bal.Acc: {gb_res['balanced_accuracy']:.3f} | AUC: {gb_res['auc']:.3f}")
    print(f"  Recall(Fail): {gb_res['recall_fail']:.3f} | F1(Fail): {gb_res['f1_fail']:.3f}")
    
    # ==============================
    # SVM
    # ==============================
    svm_model = SVC(
        kernel='rbf', C=1.0, gamma='scale',
        class_weight='balanced', probability=True, random_state=42
    )
    svm_model.fit(X_train_scaled, y_train)
    
    y_pred_svm = svm_model.predict(X_test_scaled)
    y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]
    
    results_all['SVM'].append({
        'test_year': int(test_year),
        'accuracy': accuracy_score(y_test, y_pred_svm),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred_svm),
        'auc': roc_auc_score(y_test, y_prob_svm),
        'recall_pass': recall_score(y_test, y_pred_svm, pos_label=1),
        'f1_pass': f1_score(y_test, y_pred_svm, pos_label=1),
        'recall_fail': recall_score(y_test, y_pred_svm, pos_label=0),
        'f1_fail': f1_score(y_test, y_pred_svm, pos_label=0),
    })
    
    svm_res = results_all['SVM'][-1]
    print(f"\nSVM:")
    print(f"  Acc: {svm_res['accuracy']:.3f} | Bal.Acc: {svm_res['balanced_accuracy']:.3f} | AUC: {svm_res['auc']:.3f}")
    print(f"  Recall(Fail): {svm_res['recall_fail']:.3f} | F1(Fail): {svm_res['f1_fail']:.3f}")

print(f"\n{'='*70}")
print("CROSS-VALIDATION COMPLETE")
print(f"{'='*70}")


TEST YEAR: 2014
Train size:  912 | Test size: 242
Train fail rate: 24.2%
Test fail rate:  19.0%

Logistic Regression:
  Acc: 0.822 | Bal.Acc: 0.716 | AUC: 0.780
  Recall(Fail): 0.543 | F1(Fail): 0.538

Random Forest:
  Acc: 0.826 | Bal.Acc: 0.718 | AUC: 0.768
  Recall(Fail): 0.543 | F1(Fail): 0.543

Gradient Boosting:
  Acc: 0.839 | Bal.Acc: 0.701 | AUC: 0.782
  Recall(Fail): 0.478 | F1(Fail): 0.530

SVM:
  Acc: 0.814 | Bal.Acc: 0.711 | AUC: 0.760
  Recall(Fail): 0.543 | F1(Fail): 0.526

TEST YEAR: 2015
Train size:  911 | Test size: 243
Train fail rate: 22.8%
Test fail rate:  24.3%

Logistic Regression:
  Acc: 0.708 | Bal.Acc: 0.680 | AUC: 0.754
  Recall(Fail): 0.627 | F1(Fail): 0.510

Random Forest:
  Acc: 0.741 | Bal.Acc: 0.691 | AUC: 0.742
  Recall(Fail): 0.593 | F1(Fail): 0.526

Gradient Boosting:
  Acc: 0.737 | Bal.Acc: 0.573 | AUC: 0.743
  Recall(Fail): 0.254 | F1(Fail): 0.319

SVM:
  Acc: 0.712 | Bal.Acc: 0.689 | AUC: 0.748
  Recall(Fail): 0.644 | F1(Fail): 0.521

TEST YEAR: 20

## STEP 9: Summary Comparison - Before vs After

In [37]:
print("\n" + "="*70)
print("SUMMARY: REMEDIATED FEATURES (AFTER FIX)")
print("="*70)

summary_data = {}

for model_name, results_list in results_all.items():
    df = pd.DataFrame(results_list)
    
    summary_data[model_name] = {
        'Accuracy': f"{df['accuracy'].mean():.3f} ± {df['accuracy'].std():.3f}",
        'Bal.Acc': f"{df['balanced_accuracy'].mean():.3f} ± {df['balanced_accuracy'].std():.3f}",
        'AUC': f"{df['auc'].mean():.3f} ± {df['auc'].std():.3f}",
        'Recall(Fail)': f"{df['recall_fail'].mean():.3f} ± {df['recall_fail'].std():.3f}",
        'F1(Fail)': f"{df['f1_fail'].mean():.3f} ± {df['f1_fail'].std():.3f}",
    }

summary_df = pd.DataFrame(summary_data).T
print("\nMean ± Std across 5 folds (2014-2018):")
print(summary_df.to_string())


SUMMARY: REMEDIATED FEATURES (AFTER FIX)

Mean ± Std across 5 folds (2014-2018):
                          Accuracy        Bal.Acc            AUC   Recall(Fail)       F1(Fail)
Logistic Regression  0.763 ± 0.046  0.731 ± 0.053  0.809 ± 0.045  0.669 ± 0.191  0.560 ± 0.046
Random Forest        0.758 ± 0.043  0.710 ± 0.042  0.782 ± 0.027  0.620 ± 0.177  0.536 ± 0.049
Gradient Boosting    0.785 ± 0.046  0.676 ± 0.083  0.799 ± 0.037  0.473 ± 0.220  0.486 ± 0.132
SVM                  0.760 ± 0.039  0.721 ± 0.046  0.782 ± 0.033  0.648 ± 0.170  0.549 ± 0.046


## STEP 10: Save Results

In [38]:
# Save individual model results
for model_name, results_list in results_all.items():
    df = pd.DataFrame(results_list)
    filename = f"remediation_{model_name.lower().replace(' ', '_')}_loyocv_results.csv"
    df.to_csv(os.path.join(RESULTS_DIR, filename), index=False)
    print(f"Saved: {filename}")

# Save summary statistics
summary_df.to_csv(os.path.join(RESULTS_DIR, "remediation_summary_all_models.csv"))
print(f"\nSaved: remediation_summary_all_models.csv")

# Export fixed features for inspection
X_new.to_csv(os.path.join(RESULTS_DIR, "features_after_zero_ambiguity_fix.csv"), index=False)
print(f"Saved: features_after_zero_ambiguity_fix.csv")

Saved: remediation_logistic_regression_loyocv_results.csv
Saved: remediation_random_forest_loyocv_results.csv
Saved: remediation_gradient_boosting_loyocv_results.csv
Saved: remediation_svm_loyocv_results.csv

Saved: remediation_summary_all_models.csv
Saved: features_after_zero_ambiguity_fix.csv


## STEP 11: Comparison with Original Baseline

In [39]:

# STEP 11: Comparison with Original Baseline (computed live)
print("\n" + "="*70)
print("BEFORE vs AFTER: RECALL(FAIL) COMPARISON")
print("="*70)

# --- Compute baseline on original ambiguous features ---
print("\nComputing baseline (original ambiguous features)...")

# Original feature set (X_old) is still in memory from earlier
results_original = {
    'Logistic Regression': [],
    'Random Forest': [],
    'Gradient Boosting': [],
    'SVM': []
}

for test_year in test_years:
    test_mask = year_col == test_year
    train_mask = year_col != test_year   # true LOYOCV

    X_tr = X_old.loc[train_mask].copy()
    X_te = X_old.loc[test_mask].copy()
    y_tr = y.loc[train_mask].copy()
    y_te = y.loc[test_mask].copy()

    # Logistic Regression (default params, threshold 0.50)
    scaler = StandardScaler()
    Xtr_sc = scaler.fit_transform(X_tr)
    Xte_sc = scaler.transform(X_te)
    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    lr.fit(Xtr_sc, y_tr)
    yp = lr.predict_proba(Xte_sc)[:, 1]
    yh = (yp >= 0.50).astype(int)
    results_original['Logistic Regression'].append({
        'test_year': int(test_year),
        'recall_fail': recall_score(y_te, yh, pos_label=0)
    })

    # Random Forest
    rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    yp = rf.predict_proba(X_te)[:, 1]
    yh = (yp >= 0.50).astype(int)
    results_original['Random Forest'].append({
        'test_year': int(test_year),
        'recall_fail': recall_score(y_te, yh, pos_label=0)
    })

    # Gradient Boosting
    gb = GradientBoostingClassifier(random_state=42)
    gb.fit(X_tr, y_tr)
    yp = gb.predict_proba(X_te)[:, 1]
    yh = (yp >= 0.50).astype(int)
    results_original['Gradient Boosting'].append({
        'test_year': int(test_year),
        'recall_fail': recall_score(y_te, yh, pos_label=0)
    })

    # SVM
    scaler_svm = StandardScaler()
    Xtr_sc = scaler_svm.fit_transform(X_tr)
    Xte_sc = scaler_svm.transform(X_te)
    svm = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
    svm.fit(Xtr_sc, y_tr)
    yp = svm.predict_proba(Xte_sc)[:, 1]
    yh = (yp >= 0.50).astype(int)
    results_original['SVM'].append({
        'test_year': int(test_year),
        'recall_fail': recall_score(y_te, yh, pos_label=0)
    })

baseline_recall = {
    m: pd.DataFrame(results_original[m])['recall_fail'].mean()
    for m in ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'SVM']
}

print("\n" + "{:<25} {:<12} {:<12} {:<10}".format("Model", "Before Fix", "After Fix", "Change"))
print("-" * 60)

for model_name in ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'SVM']:
    df = pd.DataFrame(results_all[model_name])
    after = df['recall_fail'].mean()
    before = baseline_recall[model_name]
    change = after - before
    arrow = "↑" if change > 0.01 else ("↓" if change < -0.01 else "→")
    print("{:<25} {:<12.3f} {:<12.3f} {} {:.3f}".format(
        model_name, before, after, arrow, change
    ))




BEFORE vs AFTER: RECALL(FAIL) COMPARISON

Computing baseline (original ambiguous features)...

Model                     Before Fix   After Fix    Change    
------------------------------------------------------------
Logistic Regression       0.661        0.669        → 0.008
Random Forest             0.625        0.620        → -0.004
Gradient Boosting         0.437        0.473        ↑ 0.035
SVM                       0.472        0.648        ↑ 0.177


## Summary

Ambiguous gap features were removed and inactivity was recalculated using a zero-ambiguity method. LOYOCV results confirm the remediated feature set performs at least as well as the original baseline. The cleaned feature matrix and per-model results are saved to the Results folder.

## References

- Pedregosa et al. (2011). Scikit-learn: Machine learning in Python. *JMLR*, 12, 2825-2830.
- Brodersen et al. (2010). The balanced accuracy and its posterior distribution. *ICPR*.
- Breiman, L. (2001). Random forests. *Machine Learning*, 45(1), 5-32.
- Friedman, J.H. (2001). Greedy function approximation: A gradient boosting machine. *Annals of Statistics*, 29(5), 1189-1232.
- Cortes, C., & Vapnik, V. (1995). Support-vector networks. *Machine Learning*, 20(3), 273-297.